# BTL-1: Compression-First Training
## Qwen 2.5 Coder 7B — SFT → DPO → Contrastive

**Requires:** T4/16GB VRAM (Colab free tier), ~3h
**Data:** Upload train.jsonl + eval.jsonl

In [ ]:
# 1. Install dependencies (run once, ~2min)
!pip install -q unsloth xformers trl datasets accelerate bitsandbytes
!pip install -q transformers torch sentencepiece

In [ ]:
# 2. Upload data zip
from google.colab import files
import zipfile, io
print('Upload btl-data.zip')
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
            z.extractall('/content')
        print(f'Extracted {name} to /content/')

In [ ]:
# 3. Write train.py
%%writefile /content/train.py
import json, os, random, math, sys
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from unsloth import FastLanguageModel

PROJECT_ROOT = Path('/content')
TRAIN_PATH = os.environ.get('BTL_TRAIN_PATH', '/content/train.jsonl')
EVAL_PATH = os.environ.get('BTL_EVAL_PATH', '/content/eval.jsonl')
BASE_MODEL = os.environ.get('BTL_BASE_MODEL', 'Qwen/Qwen2.5-Coder-7B-Instruct')
OUTPUT_DIR = Path(os.environ.get('BTL_OUTPUT_DIR', '/content/artifacts'))
DEVICE = os.environ.get('BTL_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')

LR = float(os.environ.get('BTL_LR', '2e-4'))
WD = float(os.environ.get('BTL_WD', '0.0'))
WARMUP = float(os.environ.get('BTL_WARMUP', '0.05'))
MAX_GRAD = float(os.environ.get('BTL_MAX_GRAD', '1.0'))
MAX_LEN = int(os.environ.get('BTL_MAX_LEN', '768'))
BATCH = int(os.environ.get('BTL_BATCH', '2'))
GRAD_ACCUM = int(os.environ.get('BTL_GRAD_ACCUM', '8'))
EPOCHS = int(os.environ.get('BTL_EPOCHS', '1'))
PREF_BETA = float(os.environ.get('BTL_PREF_BETA', '0.1'))
CONTRASTIVE_MARGIN = float(os.environ.get('BTL_CONTRASTIVE_MARGIN', '0.5'))
SFT_LIMIT = int(os.environ.get('BTL_SFT_LIMIT', '20000'))
DPO_LIMIT = int(os.environ.get('BTL_DPO_LIMIT', '7500'))
CONTRASTIVE_LIMIT = int(os.environ.get('BTL_CONTRASTIVE_LIMIT', '1500'))
CONTRASTIVE_SOURCE_GROUPS = int(os.environ.get('BTL_CONTRASTIVE_SOURCE_GROUPS', '800'))
DPO_SOURCE_GROUPS = int(os.environ.get('BTL_DPO_SOURCE_GROUPS', '4000'))
SEED = int(os.environ.get('BTL_SEED', '42'))

LORA_R = int(os.environ.get('BTL_LORA_R', '32'))
LORA_ALPHA = int(os.environ.get('BTL_LORA_ALPHA', '64'))
LORA_DROPOUT = float(os.environ.get('BTL_LORA_DROPOUT', '0.05'))
LORA_TARGET = os.environ.get('BTL_LORA_TARGET', 'q_proj,v_proj').split(',')
FLASH_ATTN = int(os.environ.get('BTL_FLASH_ATTN', '0'))
DRY_RUN = int(os.environ.get('BTL_DRY_RUN', '0'))

random.seed(SEED)
torch.manual_seed(SEED)

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except: pass
    print(f'  loaded {len(rows)} rows from {path}')
    return rows

def count_variants(rows):
    c = defaultdict(int)
    for r in rows:
        v = r.get('provenance', {}).get('variant', 'unknown')
        c[v] += 1
    return dict(c)

print('='*60)
print('BTL-1 Compression Pipeline')
print(f'Model: {BASE_MODEL}')
print(f'Device: {DEVICE}')
print(f'Max len: {MAX_LEN}, Batch: {BATCH}, Grad accum: {GRAD_ACCUM}')
print(f'LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'Flash Attn 2: {bool(FLASH_ATTN)}')
print(f'SFT limit: {SFT_LIMIT}, DPO limit: {DPO_LIMIT}, Contrastive limit: {CONTRASTIVE_LIMIT}')
print('='*60)

print('\nLoading data...')
train_rows = load_jsonl(TRAIN_PATH)
eval_rows = load_jsonl(EVAL_PATH)
print(f'  train variants: {count_variants(train_rows)}')
print(f'  eval variants: {count_variants(eval_rows)}')

print('\nLoading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print('Loading model with Unsloth...')
attn = 'flash_attention_2' if FLASH_ATTN else 'sdpa'
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_LEN,
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
        load_in_4bit=True,
        attn_implementation=attn,
        device_map='auto',
    )
except Exception as e:
    print(f'  Unsloth load failed ({e}), falling back to manual load')
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )

print('Configuring LoRA...')
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET,
    use_rslora=True,
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

print(f'  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def tokenize(msgs, max_len=MAX_LEN):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    tok = tokenizer(text, truncation=True, max_length=max_len, padding=False, return_tensors='pt')
    return tok

class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, batch):
        input_ids = [b['input_ids'].squeeze(0) for b in batch]
        attn_mask = [b['attention_mask'].squeeze(0) for b in batch]
        padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=self.pad_id)
        mask = torch.nn.utils.rnn.pad_sequence(attn_mask, batch_first=True, padding_value=0)
        return {'input_ids': padded, 'attention_mask': mask}

collator = Collator(tokenizer.pad_token_id)

def build_sft_dataset(rows):
    ds = []
    for r in rows[:SFT_LIMIT]:
        msgs = r.get('messages', [])
        if not msgs: continue
        tok = tokenize(msgs)
        if tok['input_ids'].shape[1] > MAX_LEN: continue
        ds.append({'input_ids': tok['input_ids'], 'attention_mask': tok['attention_mask']})
    print(f'  SFT dataset: {len(ds)} examples')
    return ds

def build_dpo_dataset(rows):
    by_tid = defaultdict(list)
    for r in rows:
        tid = r.get('provenance', {}).get('template_id', 'unknown')
        by_tid[tid].append(r)
    groups = [g for g in by_tid.values() if len(g) >= 2]
    random.shuffle(groups)
    selected = groups[:DPO_SOURCE_GROUPS]
    ds = []
    for g in selected:
        pos = [r for r in g if r.get('provenance', {}).get('variant') == 'verbose']
        neg = [r for r in g if r.get('provenance', {}).get('variant') == 'negative']
        if not pos or not neg: continue
        pos_tok = tokenize(random.choice(pos)['messages'])
        neg_tok = tokenize(random.choice(neg)['messages'])
        if pos_tok['input_ids'].shape[1] > MAX_LEN or neg_tok['input_ids'].shape[1] > MAX_LEN:
            continue
        ds.append({'chosen': pos_tok, 'rejected': neg_tok})
    print(f'  DPO dataset: {len(ds)} pairs')
    return ds[:DPO_LIMIT]

def build_contrastive_dataset(rows):
    by_family = defaultdict(list)
    for r in rows:
        family = r.get('provenance', {}).get('source_family', 'unknown')
        if r.get('provenance', {}).get('variant') == 'verbose':
            by_family[family].append(r)
    families = [g for g in by_family.values() if len(g) >= 2]
    random.shuffle(families)
    selected = families[:CONTRASTIVE_SOURCE_GROUPS]
    ds = []
    for g in selected:
        if len(g) < 2: continue
        a, b = random.sample(g, 2)
        a_tok = tokenize(a['messages'])
        b_tok = tokenize(b['messages'])
        if a_tok['input_ids'].shape[1] > MAX_LEN or b_tok['input_ids'].shape[1] > MAX_LEN:
            continue
        ds.append({'anchor': a_tok, 'positive': b_tok, 'label': torch.tensor(1.0)})
    print(f'  Contrastive dataset: {len(ds)} pairs')
    return ds[:CONTRASTIVE_LIMIT]

print('\n=== STAGE 1: SFT ===')
sft_data = build_sft_dataset(train_rows)
if DRY_RUN:
    print('DRY RUN - skipping training')
    sys.exit(0)

sft_loader = DataLoader(sft_data, batch_size=BATCH, shuffle=True, collate_fn=collator)
sft_optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sft_sched = get_cosine_schedule_with_warmup(sft_optim, int(len(sft_loader) * WARMUP / GRAD_ACCUM), len(sft_loader) // GRAD_ACCUM)

model.train()
for epoch in range(EPOCHS):
    total = 0
    for step, batch in enumerate(sft_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch, labels=batch['input_ids'])
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        total += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
            sft_optim.step()
            sft_sched.step()
            sft_optim.zero_grad()
        if (step + 1) % (GRAD_ACCUM * 10) == 0:
            print(f'  SFT epoch {epoch+1} step {step+1}/{len(sft_loader)} loss: {loss.item()*GRAD_ACCUM:.4f}')
    print(f'  SFT epoch {epoch+1} avg loss: {total / len(sft_loader):.4f}')

(OUTPUT_DIR / 'sft').mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR / 'sft'))
print('SFT adapter saved')

del sft_loader, sft_optim, sft_sched, sft_data

print('\n=== STAGE 2: DPO ===')
dpo_data = build_dpo_dataset(train_rows)
if len(dpo_data) == 0:
    print('  No DPO pairs found (need both verbose + negative variants)')
else:
    from torch.utils.data import DataLoader as DL
    dpo_loader = DL(dpo_data, batch_size=BATCH, shuffle=True)
    dpo_optim = torch.optim.AdamW(model.parameters(), lr=LR * 0.5, weight_decay=WD)
    dpo_sched = get_cosine_schedule_with_warmup(dpo_optim, int(len(dpo_loader) * WARMUP / GRAD_ACCUM), len(dpo_loader) // GRAD_ACCUM)

    for epoch in range(EPOCHS):
        total = 0
        for step, batch in enumerate(dpo_loader):
            pos = {k: v.to(DEVICE) for k, v in batch['chosen'].items()}
            neg = {k: v.to(DEVICE) for k, v in batch['rejected'].items()}
            pos_logits = model(**pos, labels=pos['input_ids']).logits
            neg_logits = model(**neg, labels=neg['input_ids']).logits
            pos_loss = F.cross_entropy(pos_logits.view(-1, pos_logits.size(-1)), pos['input_ids'].view(-1), reduction='mean')
            neg_loss = F.cross_entropy(neg_logits.view(-1, neg_logits.size(-1)), neg['input_ids'].view(-1), reduction='mean')
            loss = -F.logsigmoid(PREF_BETA * (neg_loss - pos_loss)).mean() / GRAD_ACCUM
            loss.backward()
            total += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
                dpo_optim.step()
                dpo_sched.step()
                dpo_optim.zero_grad()
            if (step + 1) % (GRAD_ACCUM * 10) == 0:
                print(f'  DPO epoch {epoch+1} step {step+1}/{len(dpo_loader)} loss: {loss.item()*GRAD_ACCUM:.4f}')
        print(f'  DPO epoch {epoch+1} avg loss: {total / len(dpo_loader):.4f}')

    (OUTPUT_DIR / 'dpo').mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(OUTPUT_DIR / 'dpo'))
    print('DPO adapter saved')

del dpo_loader, dpo_optim, dpo_sched, dpo_data

print('\n=== STAGE 3: Contrastive ===')
con_data = build_contrastive_dataset(train_rows)
if len(con_data) == 0:
    print('  No contrastive pairs found')
else:
    con_loader = DL(con_data, batch_size=BATCH, shuffle=True)
    con_optim = torch.optim.AdamW(model.parameters(), lr=LR * 0.3, weight_decay=WD)
    con_sched = get_cosine_schedule_with_warmup(con_optim, int(len(con_loader) * WARMUP / GRAD_ACCUM), len(con_loader) // GRAD_ACCUM)

    def contrastive_loss(z1, z2, label, margin=CONTRASTIVE_MARGIN):
        sim = F.cosine_similarity(z1, z2)
        pos = label * (1 - sim).pow(2)
        neg = (1 - label) * F.relu(sim - margin).pow(2)
        return (pos + neg).mean()

    for epoch in range(EPOCHS):
        total = 0
        for step, batch in enumerate(con_loader):
            anc = {k: v.to(DEVICE) for k, v in batch['anchor'].items()}
            pos = {k: v.to(DEVICE) for k, v in batch['positive'].items()}
            lbl = batch['label'].to(DEVICE)
            anc_out = model(**anc).logits
            pos_out = model(**pos).logits
            anc_pool = anc_out.mean(dim=1)
            pos_pool = pos_out.mean(dim=1)
            loss = contrastive_loss(anc_pool, pos_pool, lbl) / GRAD_ACCUM
            loss.backward()
            total += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
                con_optim.step()
                con_sched.step()
                con_optim.zero_grad()
            if (step + 1) % (GRAD_ACCUM * 10) == 0:
                print(f'  Contrastive epoch {epoch+1} step {step+1}/{len(con_loader)} loss: {loss.item()*GRAD_ACCUM:.4f}')
        print(f'  Contrastive epoch {epoch+1} avg loss: {total / len(con_loader):.4f}')

    (OUTPUT_DIR / 'contrastive').mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(OUTPUT_DIR / 'contrastive'))
    print('Contrastive adapter saved')

print('\n=== Done ===')
print(f'Final adapters saved to {OUTPUT_DIR}')

In [ ]:
# 4. Run training (~2-3h on T4)
import os
os.environ.update({
    'BTL_TRAIN_PATH': '/content/train.jsonl',
    'BTL_EVAL_PATH': '/content/eval.jsonl',
    'BTL_MAX_LEN': '768',
    'BTL_BATCH': '2',
    'BTL_GRAD_ACCUM': '8',
    'BTL_FLASH_ATTN': '0',
    'BTL_SFT_LIMIT': '15000',
    'BTL_DPO_LIMIT': '5000',
    'BTL_CONTRASTIVE_LIMIT': '1500',
    'BTL_LORA_R': '32',
    'BTL_LORA_ALPHA': '64',
    'BTL_DPO_SOURCE_GROUPS': '2000',
    'BTL_CONTRASTIVE_SOURCE_GROUPS': '800',
})

!python /content/train.py

In [ ]:
# 5. Download trained adapter
from google.colab import files
import zipfile, os

with zipfile.ZipFile('/content/adapter.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk('/content/artifacts'):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), '/content/artifacts'))

files.download('/content/adapter.zip')